# LLM Law Learning Machine: pipeline completo en Colab

Corre **el repo** (el grafo de `src/`) de punta a punta: trae el código, monta el corpus y los índices desde Drive,
levanta el decoder con Ollama, genera las respuestas con `main.py` y las mide con el `evaluate.py` oficial.

**Lo que tiene que estar en Drive** (`MyDrive/hackathon_vectorial`, la misma carpeta de siempre):
- `corpus.db` (la de los offsets, la que dejó `exportar_entrega.py --guardar-offsets`) o `corpus_v1.zip`,
- `indices/index_sin_sentencias/` e `indices/index_juris/` (con `dense.faiss`, `bm25/`, `chunk_ids.json`, `info.json`).

**Secretos de Colab** (ícono de la llave, con *Acceso al notebook* activado):
- `GITHUB_TOKEN`: un token de GitHub con permiso de lectura al repo (el repo es privado).
- `OPENROUTER_API_KEY`: la llave del juez, solo para `evaluate.py --ragas`. No se guarda en ningún archivo.

**Entorno:** *Entorno de ejecución → Cambiar tipo → GPU L4* (o T4), luego *Ejecutar todo*.
La primera vez baja el modelo (~9 GB) y lo deja en Drive; las siguientes arranca en segundos.

In [ ]:
# ===== Configuracion =====
REPO = "NBaqueroO/LLM-Law-Learning-Machine"
RAMA = "integracion"                # la rama que se quiere correr (sale de graph_development)
MODELO = "qwen3:8b-q8_0"            # Qwen3-8B en Q8: 47,05/80 en sample_50 (Q4 = "qwen3:8b": 44,36)
CONTEXTO = 8192                     # tokens; Ollama trae 4096 por defecto y 10 pasajes no caben
PARALELO = 4                        # preguntas a la vez en la GPU (sube si sobra memoria)
SPLIT = "sample"                    # "sample" para medir; "test" el sabado
ARGS_EXTRA = ["--concurrencia", str(PARALELO), "--sin-evaluar"]   # el sabado, en cada GPU: + ["--parte", "1/2"] o ["--parte", "2/2"]
USAR_JUEZ = True                    # evaluate.py --ragas (gasta la llave)
CARPETA = "/content/drive/MyDrive/hackathon_vectorial"

In [ ]:
# ===== Drive, repo y dependencias =====
import os, sys, json, glob, shutil, sqlite3, subprocess, time, zipfile
from pathlib import Path
os.environ["JAX_PLATFORMS"] = "cpu"   # bm25s importa JAX en Colab y JAX se queda con el 75 % de la GPU

from google.colab import drive, userdata
drive.mount("/content/drive")
WORK = Path(CARPETA)
RAPIDO = Path("/content/rapido"); RAPIDO.mkdir(exist_ok=True)

import base64
token = userdata.get("GITHUB_TOKEN").strip()
RAIZ = Path("/content") / REPO.split("/")[1]
url = f"https://github.com/{REPO}.git"
# el token va en una variable de entorno de git, nunca en la URL ni en el comando: asi no sale en errores
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
genv = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}

def git(*args):
    r = subprocess.run(["git", *args], env=genv, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git {args[0]} fallo:\n{r.stderr.replace(token, '***')}\n"
                           "Si dice 'not found' o 'Authentication failed': el token debe ser *classic* con permiso "
                           "'repo' (los github_pat_ no ven repos privados de otra persona).")
    return r.stdout

if (RAIZ / ".git").exists():
    git("-C", str(RAIZ), "fetch", "-q", "origin", RAMA)
    git("-C", str(RAIZ), "checkout", "-q", "-B", RAMA, f"origin/{RAMA}")
else:
    git("clone", "-q", "-b", RAMA, url, str(RAIZ))
print(subprocess.run(["git", "-C", str(RAIZ), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(RAIZ / "requirements.txt"),
                "-r", str(RAIZ / "scripts" / "requirements-evaluador.txt")], check=True)
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
print("repo listo en", RAIZ)

In [ ]:
# ===== corpus.db e indices =====
def db_sana(p):
    try:
        con = sqlite3.connect(f"file:{p}?mode=ro", uri=True)
        ok = con.execute("PRAGMA quick_check").fetchone()[0] == "ok"
        con.close()
        return ok
    except sqlite3.Error:
        return False

DB = RAPIDO / "corpus.db"
# Se copia una vez por sesion al disco local (SQLite sobre Drive es lento). La revision se hace
# sobre la copia local: revisarla en Drive obligaba a leer los GB dos veces.
if not (DB.exists() and db_sana(DB)):
    tmp = RAPIDO / "corpus.db.tmp"
    t = time.time()
    if (WORK / "corpus.db").exists():
        print(f"copiando corpus.db de Drive ({(WORK / 'corpus.db').stat().st_size / 1e9:.1f} GB)...")
        shutil.copyfile(WORK / "corpus.db", tmp)
    else:
        z = WORK / "corpus_v1.zip"
        assert z.exists(), f"No encuentro {WORK / 'corpus.db'} ni {z}"
        print(f"descomprimiendo {z}...")
        with zipfile.ZipFile(z) as zz, zz.open(next(n for n in zz.namelist() if n.endswith("corpus.db"))) as src, \
                open(tmp, "wb") as dst:
            shutil.copyfileobj(src, dst, 1 << 24)
    os.replace(tmp, DB)
    print(f"listo en {(time.time() - t) / 60:.1f} min")
assert db_sana(DB), f"{DB} esta danada: borra corpus.db de Drive y vuelve a subirla"

def indice(nombre):
    hall = sorted(glob.glob(str(WORK / "**" / nombre / "dense.faiss"), recursive=True))
    assert hall, f"No encuentro {nombre}/dense.faiss dentro de {WORK}"
    return Path(hall[0]).parent

os.environ["CORPUS_DB"] = str(DB)
os.environ["INDEX_NORMAS"] = str(indice("index_sin_sentencias"))
os.environ["INDEX_JURIS"] = str(indice("index_juris"))
# index_manifest.json (sha256 de corpus.db y de los indices): se arma una vez y queda en Drive
os.environ["INDEX_MANIFEST"] = str(Path(os.environ["INDEX_NORMAS"]).parent / "index_manifest.json")
if not Path(os.environ["INDEX_MANIFEST"]).exists():
    print("armando index_manifest.json (una sola vez, lee todos los archivos)...")
    subprocess.run([sys.executable, "-m", "src.indexing.build_index", "--solo-manifiesto", "--db", str(DB)], check=True)
con = sqlite3.connect(DB)
sin_offsets = con.execute("SELECT COUNT(*) FROM chunks WHERE inicio IS NULL").fetchone()[0]
con.close()
print("corpus.db:", DB, "\nindices:", os.environ["INDEX_NORMAS"], "+", os.environ["INDEX_JURIS"])
if sin_offsets:
    print(f"OJO: {sin_offsets} fragmentos sin inicio/fin. Para medir da igual; para la entrega sube a Drive "
          "la corpus.db de exportar_entrega.py --guardar-offsets.")

In [ ]:
# ===== Decoder: Ollama con el modelo guardado en Drive =====
if shutil.which("ollama") is None:
    # el instalador de Ollama ahora baja un .tar.zst: sin zstd falla con "exit status 1"
    subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw", shell=True,
                   stdout=subprocess.DEVNULL)
    r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
    if r.returncode or shutil.which("ollama") is None:
        raise RuntimeError("no se pudo instalar Ollama:\n" + (r.stderr or r.stdout)[-2000:])
os.environ["OLLAMA_MODELS"] = str(WORK / "ollama")         # el modelo se baja una vez y queda en Drive
os.environ["OLLAMA_CONTEXT_LENGTH"] = str(CONTEXTO)
os.environ["OLLAMA_NUM_PARALLEL"] = str(PARALELO)
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"                     # que no descargue el modelo entre preguntas
import urllib.request

def ollama_vivo():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        return True
    except Exception:
        return False

if not ollama_vivo():
    subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_vivo():
            break
        time.sleep(1)
assert ollama_vivo(), "Ollama no arranco: mira /content/ollama.log"
subprocess.run(["ollama", "pull", MODELO], check=True)

# lo que lee src/config.py
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_MODELO"] = MODELO

from src.generation.llm_engine import texto_libre
t = time.time()
print(texto_libre("Responde en una frase.", "¿Qué norma es el Código General del Proceso?"), f"({time.time() - t:.1f} s)")

In [ ]:
# ===== Pruebas rapidas del repo (con el corpus real) =====
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   capture_output=True, text=True)
print(r.stdout[-3000:] or r.stderr[-3000:])

In [ ]:
# ===== La arquitectura: el grafo compilado y el recorrido de una pregunta =====
# El diagrama sale del grafo que de verdad corre (src/graph/workflow.py), no de un dibujo aparte.
# tests/test_arquitectura.py falla si alguna arista deja de coincidir con el diagrama del equipo.
from IPython.display import Image, display
from src.graph.workflow import construir_grafo
try:
    display(Image(construir_grafo().get_graph().draw_mermaid_png()))
except Exception as e:   # sin internet para mermaid.ink: el texto se pega en mermaid.live
    print(construir_grafo().get_graph().draw_mermaid(), "\n(", e, ")")

# Una pregunta real, nodo por nodo, con lo que cada nodo deja en el estado
primera = json.loads(open("data/sample_50.jsonl", encoding="utf-8").readline())["id"]
correr = lambda args: subprocess.run([sys.executable, "-u", *map(str, args)])
correr(["main.py", "--split", "sample", "--ids", primera, "--ruta"])

In [ ]:
# ===== Correr el grafo: main.py (si se corta, volver a correr la celda sigue donde iba) =====
def correr(args):
    p = subprocess.Popen([sys.executable, "-u", *map(str, args)], text=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, encoding="utf-8", errors="replace")
    try:
        for linea in p.stdout:
            print(linea, end="")
        return p.wait()
    finally:
        if p.poll() is None:
            p.kill()

t = time.time()
cod = correr(["main.py", "--split", SPLIT, *ARGS_EXTRA])
print(f"\nmain.py termino con codigo {cod} en {(time.time() - t) / 60:.1f} min")

SALIDA = Path("outputs") / f"{'sample_50' if SPLIT == 'sample' else 'submissions'}.jsonl"
hallados = sorted((p for p in Path("outputs").glob("*.jsonl") if not p.name.endswith(".trazas.jsonl")),
                  key=lambda p: p.stat().st_mtime) if Path("outputs").exists() else []
if not SALIDA.exists() and hallados:
    SALIDA = hallados[-1]
assert SALIDA.exists(), "main.py no dejo ningun .jsonl en outputs/ (¿ya esta implementado?)"
n = sum(1 for _ in open(SALIDA, encoding="utf-8"))
print(f"{SALIDA}: {n} respuestas")
copia = WORK / "entregas_grafo" / f"{time.strftime('%Y%m%d_%H%M')}_{SALIDA.name}"
copia.parent.mkdir(exist_ok=True)
shutil.copy(SALIDA, copia)
print("copia en Drive:", copia)

In [ ]:
# ===== Evaluador oficial =====
env = dict(os.environ)
args = [sys.executable, "scripts/evaluate.py", "--submission", str(SALIDA), "--split", SPLIT,
        "--out", str(WORK / "entregas_grafo" / (copia.stem + ".eval.json"))]
if USAR_JUEZ and SPLIT == "sample":
    env["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    args.append("--ragas")
r = subprocess.run(args, env=env, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout[-6000:] or r.stderr[-6000:])
print("Linea base (agentes.ipynb, simple_8b_q8_k8): 47,05/80")